# DPVO у Colab (T4) — збірка, прогін TUM і УКУ

## A. Збірка DPVO (dpvo/01)

In [ ]:
# A1. GPU, Python, torch, CUDA
import subprocess, sys, platform
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip())
print("python", platform.python_version())
try:
    import torch
    print("torch", torch.__version__, "| torch CUDA", torch.version.cuda, "| cuda available:", torch.cuda.is_available())
except ImportError:
    print("torch not installed")
print(subprocess.run("nvcc --version | tail -n 2", shell=True, capture_output=True, text=True).stdout.strip())
assert "T4" in subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout, "Expected a T4 GPU runtime (otherwise the run is not comparable)"

In [ ]:
# A2. Mount Google Drive (persistent storage: exported UCU data, results)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/slam_dpvo"
import os
os.makedirs(DRIVE, exist_ok=True)
print("Drive dir:", DRIVE)

In [ ]:
# A3. Get our repo (private) using GITHUB_TOKEN from Colab Secrets. The token is never printed.
import os, subprocess
from google.colab import userdata

REPO_DIR = "/content/ai_project_v.piter"
REPO_BRANCH = "main"   # change if the run wrapper lives on another branch
PUBLIC_URL = "https://github.com/oToTak/ai_project_v.piter.git"

def run_hidden(cmd, secret):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(secret, "***")
    if r.returncode:
        raise RuntimeError(out)
    return out

token = userdata.get("GITHUB_TOKEN")  # raises if the secret is missing or notebook access is off
if os.path.isdir(REPO_DIR):
    run_hidden(["git", "-C", REPO_DIR, "remote", "set-url", "origin", PUBLIC_URL.replace("https://", f"https://x-access-token:{token}@")], token)
    run_hidden(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], token)
    run_hidden(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], token)
    run_hidden(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], token)
else:
    run_hidden(["git", "clone", "--branch", REPO_BRANCH, PUBLIC_URL.replace("https://", f"https://x-access-token:{token}@"), REPO_DIR], token)
run_hidden(["git", "-C", REPO_DIR, "remote", "set-url", "origin", PUBLIC_URL], token)  # strip token from .git/config
del token
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{REPO_DIR}/scripts/dpvo/run_dpvo.py"), "run wrapper is not on this branch: push it to GitHub first"

In [ ]:
%%bash
# A4. DPVO @ 0ac95b6 + Eigen 3.4.0
set -e
cd /content
[ -d DPVO ] || git clone https://github.com/princeton-vl/DPVO.git
cd DPVO
git checkout 0ac95b6
git log --oneline -1
mkdir -p thirdparty
if [ ! -d thirdparty/eigen-3.4.0 ]; then
  wget -q https://gitlab.com/libeigen/eigen/-/archive/3.4.0/eigen-3.4.0.zip
  unzip -q eigen-3.4.0.zip -d thirdparty && rm eigen-3.4.0.zip
fi
ls thirdparty

In [ ]:
# A5. Python deps + build the CUDA extensions (takes several minutes; time is printed -> put it in run.md)
import time, subprocess, torch
t0 = time.time()
tv = torch.__version__.split("+")[0]
tv_short = ".".join(tv.split(".")[:2]) + ".0"
cu = "cu" + torch.version.cuda.replace(".", "")
pyg = f"https://data.pyg.org/whl/torch-{tv_short}+{cu}.html"
print("torch-scatter wheel index:", pyg)
sh = lambda c: subprocess.run(c, shell=True, check=True)
sh("pip install -q einops pypose kornia numba tqdm yacs plyfile opencv-python matplotlib")
sh(f"pip install -q torch-scatter -f {pyg} || pip install -q --no-build-isolation torch-scatter")
# Patch for recent torch (Colab ships torch >= 2.11): DPVO/lietorch pass Tensor.type() to dispatch macros;
# nvcc fails with 'no suitable conversion function from DeprecatedTypeProperties'. .scalar_type() is equivalent.
sh(r"""cd /content/DPVO && find dpvo -name '*.cu' -o -name '*.cpp' -o -name '*.cuh' -o -name '*.h' | xargs sed -i -E 's/(AT_DISPATCH_[A-Z_]*\()([A-Za-z0-9_]+)\.type\(\)/\1\2.scalar_type()/; s/\b([A-Za-z0-9_]+)\.type\(\), "/\1.scalar_type(), "/'""")
sh("cd /content/DPVO && git diff --shortstat")  # record in run.md: DPVO sources are patched
# dpvo/loop_closure has no __init__.py, so pip omits it and 'import dpvo.dpvo' fails (patchgraph imports it).
sh(r"""cd /content/DPVO && for d in $(find dpvo -type d -not -name __pycache__); do if ls $d/*.py >/dev/null 2>&1 && [ ! -f $d/__init__.py ]; then echo "adding $d/__init__.py"; touch $d/__init__.py; fi; done""")
sh("pip install -q ninja")
sh("cd /content/DPVO && TORCH_CUDA_ARCH_LIST='7.5' pip install --no-build-isolation .")
print(f"install time: {(time.time()-t0)/60:.1f} min")

In [ ]:
%%bash
# A6. Weights: models.zip only (not the ~2 GB data). Unzips dpvo.pth into /content/DPVO
set -e
cd /content/DPVO
if [ ! -f dpvo.pth ]; then
  wget -q -O models.zip "https://www.dropbox.com/scl/fo/2g6e134ruh1acmaye5bmy/AK0DY9WLZ25elq-7N7sJcB0/models.zip?rlkey=4j336vcc4h324j4p4ji3ppl0t"
  unzip -o -q models.zip
fi
ls -la dpvo.pth

In [ ]:
# A7. Check: import dpvo, CUDA, weights; print versions for run.md
import os, platform, subprocess, torch, numpy
import dpvo, dpvo.dpvo, cuda_corr, cuda_ba, lietorch_backends
assert torch.cuda.is_available(), "CUDA not available"
assert os.path.exists("/content/DPVO/dpvo.pth"), "dpvo.pth missing"
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda, "| numpy", numpy.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("DPVO commit:", subprocess.run("git -C /content/DPVO rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip())
print("import dpvo OK")

In [ ]:
%%bash
# A8. Smoke test: download TUM fr1_xyz (also used in section C), run our wrapper on the first 150 frames.
# Output goes to /content/smoke_results (not into repo results/).
set -e
cd /content/ai_project_v.piter
if [ ! -f data/tum/fr1_xyz/rgb.txt ]; then
  mkdir -p data/tum/fr1_xyz
  wget -q -O /content/fr1_xyz.tgz https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_xyz.tgz
  tar -xzf /content/fr1_xyz.tgz -C data/tum/fr1_xyz --strip-components=1 && rm /content/fr1_xyz.tgz
fi
python3 scripts/dpvo/run_dpvo.py --dataset tum --seq fr1_xyz --data-dir data/tum/fr1_xyz \
  --calib configs/dpvo/calib_tum_fr1.txt --dpvo-dir /content/DPVO --max-frames 150 --results-root /content/smoke_results
head -n 3 /content/smoke_results/dpvo/mono/tum/fr1_xyz/traj_run*.txt
cat /content/smoke_results/dpvo/mono/tum/fr1_xyz/meta.yaml

## B. Запасний варіант: torch 2.3.1 / CUDA 12.1 (ТІЛЬКИ якщо A5 або A7 впали)
Офіційна конфігурація DPVO: torch 2.3.1, cu121, numpy 1.26.4. Після виконання цієї клітинки рантайм перезапуститься —
далі запустити A3-A8 знову (A1, A2 теж, щоб знову змонтувати Drive).
Якщо запас не допоміг — записати помилку й повідомити (потрібне інше меню варіантів).


**Увага:** на Colab з Python 3.13 цей розділ не працює (для torch 2.3.1 немає коліс). Замість нього клітинка A5 патчить `.type()` -> `.scalar_type()` у джерелах DPVO і збирає з torch Colab (перевірено 24.10.2026: torch 2.11.0+cu130, CUDA 13.0).

In [ ]:
# B1. OPTIONAL fallback. Skip if section A worked.
import subprocess, os
sh = lambda c: subprocess.run(c, shell=True, check=True)
sh("pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121")
sh("pip install -q numpy==1.26.4")
sh("pip uninstall -y -q torch-scatter dpvo || true")
sh("rm -rf /content/DPVO/build")
os.kill(os.getpid(), 9)  # restart runtime

## C. TUM fr1_xyz (dpvo/03)
Потрібні виконані A1-A3 і A4-A7 (A8 уже завантажив дані). Далі — повний прогін.

In [ ]:
%%bash
# C1. Full run: results/dpvo/mono/tum/fr1_xyz/traj_run<N>.txt + meta.yaml (N = next free number)
set -e
export DPVO_DIR=/content/DPVO
cd /content/ai_project_v.piter
test -f data/tum/fr1_xyz/rgb.txt
time scripts/dpvo/run.sh tum fr1_xyz
ls results/dpvo/mono/tum/fr1_xyz
cat results/dpvo/mono/tum/fr1_xyz/meta.yaml

## D. УКУ seq01 (dpvo/04)
Потрібні `HF_TOKEN` у Secrets (умови gated-датасету прийняті) та виконані A1-A7.
`sequence-1.mcap` = 6.5 GB: качаємо на **локальний диск Colab** (`/content`, у Drive не кладемо), експорт іде потоково (rosbags),
RAM не вичерпує. Експорт (PNG, кілька GB) — теж локально, а в Drive лягає один tar; у новій сесії D1-D3 пропускаються, якщо tar уже є.

In [ ]:
%%bash
# D1. Restore exported data from Drive if present
set -e
cd /content/ai_project_v.piter
TAR=/content/drive/MyDrive/slam_dpvo/ucu_tum_seq01.tar
if [ -f "$TAR" ] && [ ! -f data/ucu_tum/seq01/rgb.txt ]; then
  mkdir -p data/ucu_tum && tar -xf "$TAR" -C data/ucu_tum && echo restored from Drive
fi
ls data/ucu_tum/seq01 2>/dev/null | head -n 5 || echo "not exported yet -> run D2-D4"

In [ ]:
# D2. Download sequence-1 from HF (gated; token from Secrets, not printed). Skip if D1 restored the data.
import os
from google.colab import userdata
if os.path.exists("/content/ai_project_v.piter/data/ucu_tum/seq01/rgb.txt"):
    print("already exported, skipping")
else:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="ucu-autonomous-ugv/ucu-slam-dataset-v1", repo_type="dataset",
                      allow_patterns=["sequence-1/*"], local_dir="/content/ai_project_v.piter/data/ucu")
    print(subprocess.run("ls -la /content/ai_project_v.piter/data/ucu/sequence-1", shell=True, capture_output=True, text=True).stdout)
    os.environ.pop("HF_TOKEN", None)

In [ ]:
%%bash
# D3. Official exporter (github.com/ucu-autonomous-ugv/ucu-slam-dataset) needs Python >= 3.12 -> separate uv venv.
set -e
if [ -f /content/ai_project_v.piter/data/ucu_tum/seq01/rgb.txt ]; then echo "already exported, skipping"; exit 0; fi
pip install -q uv
cd /content
[ -d ucu-slam-dataset ] || git clone https://github.com/ucu-autonomous-ugv/ucu-slam-dataset.git
uv venv --python 3.12 /content/venv_export
uv pip install --python /content/venv_export/bin/python -r /content/ai_project_v.piter/requirements.txt
cd /content/ucu-slam-dataset
mkdir -p /content/ai_project_v.piter/data/ucu_tum
/content/venv_export/bin/python scripts/export_tum.py /content/ai_project_v.piter/data/ucu/sequence-1 /content/ai_project_v.piter/data/ucu_tum/seq01
head -n 5 /content/ai_project_v.piter/data/ucu_tum/seq01/rgb.txt
wc -l /content/ai_project_v.piter/data/ucu_tum/seq01/rgb.txt /content/ai_project_v.piter/data/ucu_tum/seq01/groundtruth.txt

In [ ]:
%%bash
# D4. Persist the export on Drive (one tar), then free the raw mcap
set -e
TAR=/content/drive/MyDrive/slam_dpvo/ucu_tum_seq01.tar
if [ ! -f "$TAR" ]; then
  tar -cf "$TAR" -C /content/ai_project_v.piter/data/ucu_tum seq01
fi
ls -la "$TAR"
rm -rf /content/ai_project_v.piter/data/ucu/sequence-1/*.mcap

In [ ]:
%%bash
# D5. Run DPVO on seq01 (6532 frames; note the wall time for run.md)
set -e
export DPVO_DIR=/content/DPVO
cd /content/ai_project_v.piter
test -f data/ucu_tum/seq01/rgb.txt
time scripts/dpvo/run.sh ucu seq01
ls results/dpvo/mono/ucu/seq01
cat results/dpvo/mono/ucu/seq01/meta.yaml

## E. Забрати результати
Обрано **zip + Drive**, без git push з Colab (не потрібен токен із правом запису). Далі на Mac:
розпакувати zip у корінь репо (шляхи `results/dpvo/...` збережено), перевірити, закомітити.

In [ ]:
# E1. Zip results/dpvo (traj_run*.txt + meta.yaml), copy to Drive, trigger browser download
import shutil, os, time
from google.colab import files
os.chdir("/content/ai_project_v.piter")
stamp = time.strftime("%Y%m%d_%H%M")
base = f"/content/dpvo_results_{stamp}"
zp = shutil.make_archive(base, "zip", root_dir=".", base_dir="results/dpvo")
shutil.copy(zp, "/content/drive/MyDrive/slam_dpvo/")
print("saved", zp, "and a copy in Drive slam_dpvo/")
files.download(zp)